# Mesonet Regression Example
## Understanding Regression and Generalization

This is our mesonet example. The mesonet is a network of environmental monitoring stations across the state of Oklahoma. This dataset compiled several years of data from these stations. We will be using a cleaned up version of the data (some values were missing from broken sensors) to predict rainfall in both a classification sense and a regression sense.

By the end of this notebook you'll know how to:

- Configure a model for regression
- Choose an appropriate regression loss function
- Evaluate regression predictions
- Identify overfitting and underfitting
- Use regularization to improve model performance
- Use argument override files

# Regression

Up until now the only machine learning tasks we have done were classification, now we will move onto regression.

Regression focuses on predicting a numerical output rather than a class. For instance, you may want to try to predict a housing price based off things like square footage, amount of bedrooms and bathrooms, and area. This is a regression problem as you take in your input features and try to predict and exact price of the house. So classifcation is predicting a group the example belongs to while regression is computing the continuous numerical value produced by the inputs. 

For this dataset we will explore regression first, in the form of predicting rainfall down to the inch, and then turn it into a classification example.

Changing to this problem will require some new loss functions and output activation functions, we will go over options shortly. For now let's examine our dataset.

In [ ]:
import os
import sys

# Optional if you don't have the neuro path variable set up in your bashrc (Set to folder/directory above keras3_tools and zero2neuro)
# os.environ["NEURO_REPOSITORY_PATH"] = "/home/myuser/neuro"

neuro_path = os.getenv("NEURO_REPOSITORY_PATH")
assert neuro_path is not None, "Environment variable NEURO_REPOSITORY_PATH must be set to directory above zero2neuro and keras3_tools"

sys.path.append(neuro_path + '/zero2neuro/src/')

from zero2neuro import *
from parser import *

In [ ]:
parser = create_parser()

In [ ]:
import pandas as pd
df = pd.read_csv("../data/meso_data.csv")

In [ ]:
df.head()

In [ ]:
# This dataset has been downsized to be 2000 examples by 39 features.
df.shape

In [ ]:
'''
Here we have cleaned up some of the input features (removed some that had too many bad values).
However you may notice that we still have YEAR, MONTH, and DAY. 
We do not plan to use these for this example but these columns do have value in
certain contexts in machine learning, we will just ignore them during setup however.
'''
df.columns

In [ ]:
df.info()

# Data Configuration
data_config.txt has been filled out for you but feel free to open it to take a look. We will skip it in this section as it's all things we have already covered.

# Experiment Configuration
Open up experiment_config.txt

```
--experiment_name=mesonet_regression
--data_rotation=0
--loss=TODO
--metrics
mae
mse

--learning_rate=TODO
--epochs=TODO

--early_stopping
--early_stopping_monitor=val_loss
--early_stopping_patience=50

# The threshold for what we count as "not improving"
--early_stopping_min_delta=0.01

--results_path=./results
--output_file_base={args.experiment_name}_R{args.data_rotation:02d}
--save_model
--render_model

# Save the training set results
--log_training_set

# Save the validation set results
--log_validation_set

# Save the testing set results
--log_testing_set
```

Since this is a regression problem we need a new loss function. The two options recommended are either mse or mae. In terms of this problem here's a rough idea of how each one works.  

MAE
- $MAE = |\text{actual} - \text{predicted}|$
- If the actual rainfall is 10"
  - Prediction = 9" -> error = 1"
  - Prediction = 5" -> error = 5"
  - The 5" prediction is 5x worse than the 9" prediction.      

MSE
- $MSE = (\text{actual} - \text{predicted})^2$
- For the same predictions
  - Prediction = 9" -> error = 1"
  - Prediction = 5" -> error = 25"
  - Now the 5" prediction is 25x worse

So MSE punishes larger errors a lot harder than MAE does. Either one will work for this problem and it's good to try out both and compare them.

The second new thing to notice is early stopping min delta. We've talked about early stopping several times but min delta is what decides substantial learning looks like. So if the error hasn't improved by 0.01 in 50 epochs (in this example), the model will go back to when val_loss was lowest and stop the training. Feel free to edit this delta and the patience as you see fit.

# Network Configuration
```
--network_type=fully_connected
--input_shape
34
--number_hidden_units
TODO
--hidden_activation=elu
--output_shape
1
--output_activation=elup1

--dropout=???
```
Here we use a new output activation (since we want to output a numerical value). We use elup1, which elu itself goes from -1 to infinity but with elup1 it goes from 0 to infinity. The reason why we use it on this data is because negative rainfall doesn't physically make sense to try to predict. 
  
Next we have a new argument, dropout. This is a form of what we call regularization. 

## Overfitting, Underfitting, and Using Regularization
We hinted at overfitting and underfitting in the iris example but here we will formally define them and look into the techniques to fix them. 

Overfitting
- The model learns the training data too well including any random noise in the training data leading to bad generalization on unseen data.
- Is often handled by introducing regularization techniques that penalize the model to prevent it from becoming too complex.

Underfitting
- The model is too simple to capture the patterns in the data causing poor performance in training, validation, and testing sets.
- Is handled by reducing regularization techniques, making the model more complex, or adding more informative features to the dataset

There are lots of regularization techniques but for this example we will focus on two: early stopping and dropout. As soon as we started using val_loss for early stopping it became a regularization technique as it monitors for overfitting. As for dropout, it is a complex topic but to boil it down to simple terms: it randomly switches off neurons in the network to prevent the model from relying on a certain pathway. 

For this example I suggest starting out with 0.1 (10%) and going up by 5% if needed. 

# Override Argument Files
While we suggest splitting arguments into the three main files, the argument reader as mentioned in xor reads from left to right top to bottom. You can put more arguments after previous ones to "override" the past arguments. We will employ this in two ways: to switch between tasks and to have an easy way to turn on certain features like reporting. In this example we have 3: classification_override.txt, wandb.txt, and report.txt 

# A Brief Aside: Weights & Biases (OPTIONAL)
In Zero2Neuro we support integration with a web toolkit called Weights & Biases. This acts as a way to monitor experiments live while they are running and includes epoch visualizations. It is a great tool for making live decisions and giving insight into how the model is doing while training. If you are interested folllow the instructions [here](../../../docs/modules/zero2neuro/wandb.md) to get it set up and enabled and then you can pass in the wandb.txt file to turn it on for this example to try it out.

With all that said, fill out your configuration files and it's time to train (this is a much larger dataset than we have used before, it may take a little longer to train, you might decrease the epochs also).

In [ ]:
arg_string = "@network_config.txt @data_config.txt @experiment_config.txt @report.txt -v --force"
args = parser.parse_args(arg_string.split())
print(args)

In [ ]:
prepare_and_execute_experiment(args)

# Regression Results
We will do our normal epoch visualizations but for regression there's a new one to utilize: a scatterplot. 

In [ ]:
with open('results/mesonet_regression_R00_results.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file) # Grab the data from pickle

In [ ]:
print(data.keys())
print(data['history'].keys())

In [ ]:
# This code is copied from the iris example, only difference is that instead of accuracy we choose
# the loss function we didn't use (mae or mse) 

# First our epoch visualizations
# This creates a blank side by side grid for our plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss for both
axes[0].plot(data["history"]["mae"], label="Training")
axes[0].plot(data["history"]["val_mae"], label="Validation")

# Sets up our learning curve plot
axes[0].set_title("Training and Validation MAE Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (MAE)")
axes[0].legend()
axes[0].grid(alpha=0.5) 

# 
axes[1].plot(
    data["history"]["mse"],
    label="Training"
)
axes[1].plot(
    data["history"]["val_mse"],
    label="Validation"
)

axes[1].set_title("Training and Validation MSE Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss (MSE)")
axes[1].legend()
axes[1].grid(alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Let's do a scatterplot now
y_true = data["outs_testing"]
y_pred = data["predict_testing"]

plt.figure(figsize=(6,6))

plt.scatter(
    y_true,
    y_pred,
    s=8,
    alpha=0.6,
)

# Red line parameters
xmin = min(y_true.min(), y_pred.min())
xmax = max(y_true.max(), y_pred.max())

plt.plot(
    [xmin, xmax],
    [xmin, xmax],
    "r--", # This creates our red "perfect prediction" line
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("True Rainfall (in)")
plt.ylabel("Predicted Rainfall (in)")
plt.title("Testing Set Scatterplot")
plt.axis("square")
plt.legend()

plt.tight_layout()
# In case you want to save your figures, use .savefig()
# plt.savefig("file path here", dpi=300)
plt.show()


In a scatterplot all the blue dots should be exactly on the red line. As you can see a lot of the blue dots hover at zero and there's a lot of outliers from the red line. These results may be disappointing compared to our past examples, but this is a good simulation of a difficult problem. In Oklahoma it's common to go several weeks with no rain, so the dataset is naturally biased towards no rain. Despite that we can see the model still has some capability to capture a vague relationship between our features.   

Sometimes it's important to rethink the problems you can do with a dataset. So let's do that. 

# New (Old) Approach: Classification
How about instead of predicting down to an inch we just predict whether it rained or not that day? We have already got the dataset set up with a column to make this possible. All you have to do is switch this problem over to classifcation. 

```
# Use this configuration file to turn the default regression problem
#  into a classification one

# Change experiment name
--experiment_name=mesonet_categorical

# Change column where desired outputs are coming from
--data_outputs
RAIN BINARY

# Encodes the output labels
--data_columns_categorical_to_int
RAIN BINARY:NO_RAIN,RAINED

# Change the output activation function
--output_activation
TODO

# Change the loss and metrics
--loss=TODO
--metrics=TODO
```
If you've been following the examples one by one, you've already done three classification problems. Use those past examples to decide what functions you should use, and remember the new problem is now deciding between rain or no rain. 

In [ ]:
arg_string = "@network_config.txt @data_config.txt @experiment_config.txt @classification_override.txt @report.txt -v --force"
args = parser.parse_args(arg_string.split())
print(args)

In [ ]:
prepare_and_execute_experiment(args)

In [ ]:
# Load it up
with open('results/mesonet_categorical_R00_results.pkl', 'rb') as pickle_file:
    data = pickle.load(pickle_file) # Grab the data from pickle

In [ ]:
print(data.keys())
print(data['history'].keys())

In [ ]:
# Copied from iris example
# This creates a blank side by side grid for our plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss for both
axes[0].plot(data["history"]["loss"], label="Training")
axes[0].plot(data["history"]["val_loss"], label="Validation")

# Sets up our learning curve plot
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.5)

# Accuracy
axes[1].plot(
    data["history"]["binary_accuracy"],
    label="Training"
)
axes[1].plot(
    data["history"]["val_binary_accuracy"],
    label="Validation"
)

axes[1].set_title("Training and Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.5)

plt.tight_layout()
plt.show()

There is one more visualization to be done for a classification problem like this, a confusion matrix. At this point we've done three of them, so I encourage you to try your hand at creating one from scratch and evaluating the results.

In [ ]:
# Confusion Matrix Code Here